# Exploration of mask mapping — values raw, GG4 and subtype **G4C**

Goals:
- Inspect **which grayscale values** masks take tras `cv2.imdecode` (before LUT mapping).
- Compare distributions by **Excel label** (NC, G3, G4, G5).
- Focus on **GG4**: patches with `G4=1` and subtype **G4C=1** (cribriform) vs **G4C=0**.
- **GG4 pixels under the current LUT**: between patches G4C+ and G4C−, compare raw intensities only on pixels that `MASK_LUT` maps a class 2 (GG4).
- **Visualize **image + mask pairs** (grayscale and LUT/class colored).
- **Pixel-by-pixel inspection** (`inspect_mask_pixelwise`): matrices 2D grayscale and class (same LUT + GG5 cleanup as `training_conch`), table and CSV; notes about **histogram peaks** and **noise in boundaries**.

Default LUT reference (`build_mask_lut` / `DEFAULT_GG5_GRAY_MIN=170`): `[43:85)→GG3`, `[85:170)→GG4`, `[170:)→GG5`, rest NC (no `[160:170)` as GG5).

**Kernel:** project environment (`prostata_env`) if fails `openpyxl`/`cv2`.

In [ ]:
from __future__ import annotations

from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from tqdm.auto import tqdm

from sicap_mapping import CLASS_NAMES, MASK_LUT, build_mask_lut, clean_gg5_speckles
try:
    from sicap_mapping import DEFAULT_GG5_GRAY_MIN
except ImportError:
    DEFAULT_GG5_GRAY_MIN = 170



def find_project_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "partition").is_dir():
            return p
    raise FileNotFoundError("Could not find partition/; set BASE manually.")


BASE = find_project_root()
IMAGES_DIR = BASE / "images"
MASKS_DIR = BASE / "masks"
PARTITION = BASE / "partition"
RNG = np.random.default_rng(42)

print("BASE:", BASE)

In [ ]:
def list_partition_excels() -> list[Path]:
    out: list[Path] = []
    for pat in (
        "Validation/*/Train.xlsx",
        "Validation/*/Test.xlsx",
        "Test/Train.xlsx",
        "Test/Test.xlsx",
    ):
        out.extend(sorted(PARTITION.glob(pat)))
    return out


def load_partition_unique() -> pd.DataFrame:
  """One row per image_name (first appearance in Excel files)."""
    parts = []
    for xp in list_partition_excels():
        df = pd.read_excel(xp)
        df["_from"] = str(xp.relative_to(BASE))
        parts.append(df)
    all_df = pd.concat(parts, ignore_index=True)
    all_df = all_df.drop_duplicates(subset=["image_name"], keep="first")
    return all_df


meta = load_partition_unique()
meta["main_label"] = meta[["NC", "G3", "G4", "G5"]].idxmax(axis=1)
meta["g4c"] = meta["G4C"].fillna(0).astype(int)
print("Unique patches:", len(meta))
print(meta["main_label"].value_counts())
print("\nEntre rows with main_label==G4:")
g4 = meta["main_label"] == "G4"
print(meta.loc[g4, "g4c"].value_counts().sort_index())

In [ ]:
def read_mask_gray(name: str) -> np.ndarray | None:
    p = MASKS_DIR / name
    if not p.is_file():
        return None
    buf = np.fromfile(str(p), dtype=np.uint8)
    m = cv2.imdecode(buf, cv2.IMREAD_GRAYSCALE)
    return m


def read_image_rgb(name: str) -> np.ndarray | None:
    p = IMAGES_DIR / name
    if not p.is_file():
        return None
    buf = np.fromfile(str(p), dtype=np.uint8)
    bgr = cv2.imdecode(buf, cv2.IMREAD_COLOR)
    if bgr is None:
        return None
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def accumulate_hist_full_image(names: list[str], max_patches: int | None = None) -> np.ndarray:
  """Global histogram 0..255 de todos los pixels of the masks listadas."""
    h = np.zeros(256, dtype=np.float64)
    use = names
    if max_patches is not None and len(names) > max_patches:
        use = list(RNG.choice(names, size=max_patches, replace=False))
    for name in tqdm(use, desc="Masks (hist. completo)"):
        m = read_mask_gray(name)
        if m is None:
            continue
        bc = np.bincount(m.ravel(), minlength=256)
        h += bc.astype(np.float64)
    return h


def accumulate_gg4_pixels_raw(
    names: list[str],
    *,
    gg5_gray_min: int | None = None,
    mask_clean_min_gg5_area: int | None = None,
) -> np.ndarray:
  """Only pixels donde la LUT (training) mapea a GG4 (2); concatena grayscale raw."""
    g5 = DEFAULT_GG5_GRAY_MIN if gg5_gray_min is None else int(gg5_gray_min)
    ca = 16 if mask_clean_min_gg5_area is None else int(mask_clean_min_gg5_area)
    lut = build_mask_lut(g5)
    chunks: list[np.ndarray] = []
    for name in tqdm(names, desc="Pixeles GG4 (LUT=2)"):
        m = read_mask_gray(name)
        if m is None:
            continue
        mapped = lut[m].astype(np.int64)
        if ca > 0:
            mapped = clean_gg5_speckles(mapped, min_area=ca)
        raw_gg4 = m[mapped == 2]
        if raw_gg4.size:
            chunks.append(raw_gg4.ravel())
    if not chunks:
        return np.array([], dtype=np.uint8)
    return np.concatenate(chunks)


# Cortes de grayscale (GG3, GG4, inicio GG5) alineados with sicap_mapping / training
LUT_BOUNDARIES = (43, 85, DEFAULT_GG5_GRAY_MIN)

## Why there are so many peaks in the histogram

- **Canonical values per class:** when generating the mask, only a few discrete grayscale levels are used (e.g. uno by Gleason / fondo). En el histograma appear como **picos agudos**, no como una campana continua.
- **JPEG:** when saving the mask as JPEG, el codec agrupa bloques 8×8 (DCT + cuantización). This **concentrates** values around specific integers and can create **intermediate values** between two classes (sobre todo in **boundaries** between regiones o between tejido and fondo).
- **Boundaries and mapping noise:** at boundaries, a pixel may fall in a grayscale range that LUT assigns to **another class** que el vecino interior (e.g. ~82 vs ~88 near threshold 85). Eso explica parte of the **incorrect mapping near boundaries** sin que el tejido cambie de grado real.

**Goal of the 4-class setup:** the LUT only **splits the 0-255 axis** into intervals; boundary tuning may require adjusting thresholds, **suavizado morfológico** post-LUT, o reglas only on pixels de frontera — se can explorar after de ver la tabla pixel a pixel.

---

## Pixel-by-pixel inspection for un `image_name`

La función siguiente carga la mask, aplica el mapeo actual and deja las matrices **2D** (`raw` and `class`). Optionalmente genera una tabla larga `(row, column, grayscale, class)` and exporta CSV. **No imprime el array completo in consola** (sería enorme); usa `preview` o `save_csv`.

In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print


def mask_pixelwise_table(raw: np.ndarray, class_map: np.ndarray) -> pd.DataFrame:
  """Una row by pixel: row, column, grayscale, class_id, class_name."""
    h, w = raw.shape
    ri, ci = np.indices((h, w))
    cid = class_map.ravel().astype(np.int64)
    return pd.DataFrame(
        {
            "row": ri.ravel(),
            "col": ci.ravel(),
            "gray": raw.ravel().astype(np.int64),
            "class_id": cid,
            "class_name": [CLASS_NAMES[c] for c in cid],
        }
    )


def inspect_mask_pixelwise(
    image_name: str,
    *,
    save_csv: str | Path | None = None,
    preview_rows: int = 12,
    plot: bool = True,
    gg5_gray_min: int | None = None,
    mask_clean_min_gg5_area: int | None = None,
):
  """
  Loads a mask and applies the same logic as training_conch (SICAPv2Dataset):
  build_mask_lut(gg5_gray_min) and optionalmente clean_gg5_speckles (GG5
  with componentes muy pequeños → GG4).

  By default: gg5_gray_min = DEFAULT_GG5_GRAY_MIN (170);
  mask_clean_min_gg5_area = 16 (como DEFAULT_CONFIG in training_conch).
  Usa mask_clean_min_gg5_area=0 for disabler la limpieza.

  Devuelve:
  - raw_hw: (H,W) uint8, grayscale decodificado
  - class_hw: (H,W) int64, classes 0..3 tras LUT + limpieza
  """
    raw = read_mask_gray(image_name)
    if raw is None:
    print("No mask:", MASKS_DIR / image_name)
        return None

    g5 = DEFAULT_GG5_GRAY_MIN if gg5_gray_min is None else int(gg5_gray_min)
    clean_a = 16 if mask_clean_min_gg5_area is None else int(mask_clean_min_gg5_area)

    mask_lut = build_mask_lut(g5)
    class_hw = mask_lut[raw].astype(np.int64)
    if clean_a > 0:
        class_hw = clean_gg5_speckles(class_hw, min_area=clean_a)

    h, w = raw.shape
  print(f"image_name: {image_name}")
  print(f"shape (H×W): {h} × {w} = {h * w:,} pixels")
  print(f"[como training] gg5_gray_min={g5}, mask_clean_min_gg5_area={clean_a}")

  # Valores unique de grayscale and recuentos
    ug, cnt_g = np.unique(raw, return_counts=True)
    tab_gray = pd.DataFrame({"gray": ug.astype(np.int64), "count": cnt_g})
  print("\nValores de grayscale unique in la mask (toda la imagen):")
    display(tab_gray)

    uc, cnt_c = np.unique(class_hw, return_counts=True)
    tab_cls = pd.DataFrame(
        {"class_id": uc.astype(np.int64), "class_name": [CLASS_NAMES[int(c)] for c in uc], "count": cnt_c}
    )
  print("\nClases tras LUT + limpieza (recuento de pixels):")
    display(tab_cls)

    tbl = mask_pixelwise_table(raw, class_hw)
    if preview_rows and len(tbl) > 0:
    print(f"\nPrimeras {preview_rows} rows (orden row-major: col varía rápido):")
        display(tbl.head(preview_rows))
        if len(tbl) > preview_rows:
      print(f"... ({len(tbl):,} rows in total; usar save_csv for volcar todo)")

    if save_csv is not None:
        out = Path(save_csv)
        tbl.to_csv(out, index=False)
    print(f"\nCSV savedo: {out.resolve()} ({len(tbl):,} rows)")

    if plot:
        fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.2))
        axes[0].imshow(raw, cmap="gray", vmin=0, vmax=255)
        axes[0].set_title("Gris (bruto)")
        axes[1].imshow(
            class_hw,
            cmap=ListedColormap(["#222", "#4daf4a", "#377eb8", "#e41a1c"]),
            vmin=0,
            vmax=3,
        )
        axes[1].set_title("Clase 0–3 (LUT + limpieza)")
        for ax in axes:
            ax.axis("off")
        plt.suptitle(image_name[:70] + ("…" if len(image_name) > 70 else ""))
        plt.tight_layout()
        plt.show()

    return {"raw_hw": raw, "class_hw": class_hw, "table": tbl}


# Ejemplo: cambia IMAGE_NAME_DEMO by cualquier `image_name` de `meta` o pega el string completo
IMAGE_NAME_DEMO = '16B0006668_Block_Region_3_10_15_xini_69838_yini_77884.jpg'
print("IMAGE_NAME_DEMO (by default, primero of the meta):", IMAGE_NAME_DEMO)
result = inspect_mask_pixelwise(IMAGE_NAME_DEMO, preview_rows=12)
# Volcar todos los pixels a CSV (muchas rows):
# result = inspect_mask_pixelwise(IMAGE_NAME_DEMO, save_csv=BASE / "tmp_mask_pixels.csv", preview_rows=0, plot=True)


## Noise and consistency of the mapeo (LUT + limpieza)

Same pipeline que `training_conch`: `build_mask_lut(DEFAULT_GG5_GRAY_MIN)` and `clean_gg5_speckles` (área mínima 16).

- **Por mask**: porcentaje de pixels NC, GG3, GG4, GG5; se marcan masks donde alguna class appears with fracción muy pequeña (posible **noise JPEG / boundaries**), sobre todo Residual GG5.
- **Histograms**: distribution of the % of the class more débil presente; distribution of the % GG5 only donde hay GG5.
- **Excel vs mask**: tabla cruzada `main_label` (parche) versus **mayoría** in la mask (no deben coincidir siempre: el tumor can ocupar poca superficie of the parche).

Los histogramas de more abajo (por grupo Excel) usan los sames cortes in grayscale que `LUT_BOUNDARIES` (ahora with umbral GG5 = `DEFAULT_GG5_GRAY_MIN`, no 160).


In [ ]:
# Same pipeline as training_conch
GG5_M = DEFAULT_GG5_GRAY_MIN
CLEAN_A = 16
# Threshold: class marked as "present" but with very low patch fraction (posible noise / boundaries)
NOISE_PCT_THRESHOLD = 0.1  # % del parche; increase to 0.5 if there are very few cases


def map_mask_training(raw: np.ndarray) -> np.ndarray:
    lut = build_mask_lut(GG5_M)
    m = lut[raw].astype(np.int64)
    if CLEAN_A > 0:
        m = clean_gg5_speckles(m, min_area=CLEAN_A)
    return m


def per_mask_class_stats(names: list[str]) -> pd.DataFrame:
  """One row per mask: class percentages and counts after mapping."""
    rows: list[dict] = []
    for name in tqdm(names, desc="Fractions per mask"):
        raw = read_mask_gray(name)
        if raw is None:
            continue
        m = map_mask_training(raw)
        npx = m.size
        bc = np.bincount(m.ravel().astype(np.int64), minlength=4)
        pcts = 100.0 * bc / max(npx, 1)
        present = [c for c in range(4) if bc[c] > 0]
        min_pct = min(float(100.0 * bc[c] / npx) for c in present) if present else 0.0
        rows.append(
            {
                "image_name": name,
                "n_pixels": int(npx),
                "majority_class": int(np.argmax(bc)),
                **{f"pct_{CLASS_NAMES[c]}": float(pcts[c]) for c in range(4)},
                **{f"n_{CLASS_NAMES[c]}": int(bc[c]) for c in range(4)},
                "min_pct_among_present": min_pct,
            }
        )
    return pd.DataFrame(rows)


mask_per_class = per_mask_class_stats(meta["image_name"].tolist())
print("Mascaras analizadas:", len(mask_per_class))

LABEL_TO_IDX = {"NC": 0, "G3": 1, "G4": 2, "G5": 3}
ms = mask_per_class.merge(meta[["image_name", "main_label", "g4c"]], on="image_name", how="left")
ms["excel_idx"] = ms["main_label"].map(LABEL_TO_IDX)
ms["majority_name"] = ms["majority_class"].map(lambda i: CLASS_NAMES[int(i)])

# --- 1) Masks with alguna class in porcentaje muy bajo (posible noise)
print(f"\n--- Clases with 0 < pct < {NOISE_PCT_THRESHOLD}% of the parche ---")
for c in range(4):
    cn = CLASS_NAMES[c]
    col_pct = f"pct_{cn}"
    col_n = f"n_{cn}"
    hit = ms[(ms[col_n] > 0) & (ms[col_pct] < NOISE_PCT_THRESHOLD)]
  print(f" {cn}: {len(hit):,} masks")
display(
    pd.DataFrame(
        {
            "class": [CLASS_NAMES[c] for c in range(4)],
            "n_masks_sospechosas": [
                int(((ms[f"n_{CLASS_NAMES[c]}"] > 0) & (ms[f"pct_{CLASS_NAMES[c]}"] < NOISE_PCT_THRESHOLD)).sum())
                for c in range(4)
            ],
        }
    )
)

# Ejemplos: menor % mínimo between classes presentes (patches casi monocromos in class minoritaria)
show = ms.nsmallest(12, "min_pct_among_present")[
    ["image_name", "main_label", "majority_name", "min_pct_among_present"]
    + [f"pct_{CLASS_NAMES[c]}" for c in range(4)]
]
print("\n12 masks with menor 'min_pct_among_present' (minoritaria muy debil):")
display(show)

# --- 2) Distribution de min_pct_among_present (hay muchos patches casi puros in una class)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
ax0, ax1 = axes
ax0.hist(ms["min_pct_among_present"], bins=60, color="#555", edgecolor="white", linewidth=0.3)
ax0.set_xlabel("min % entre classs presentes (por mask)")
ax0.set_ylabel("conteo")
ax0.set_title("Histograma: class minoritaria mas debil")
ax0.axvline(NOISE_PCT_THRESHOLD, color="C3", ls="--", label=f"umbral {NOISE_PCT_THRESHOLD}%")
ax0.legend()

g5p = ms[ms["n_GG5"] > 0]
if len(g5p):
    ax1.hist(g5p["pct_GG5"], bins=50, color="#e41a1c", alpha=0.85, edgecolor="white", linewidth=0.3)
    ax1.set_xlabel("% pixels GG5 (only masks con GG5>0)")
    ax1.set_ylabel("conteo")
    ax1.set_title(f"Residual GG5 (n masks con GG5: {len(g5p):,})")
else:
    ax1.text(0.5, 0.5, "Ninguna mask con GG5", ha="center", va="center")
plt.tight_layout()
plt.show()

# --- 3) Excel vs mayoría in mask
ct = pd.crosstab(ms["main_label"], ms["majority_name"], margins=True)
print("\nCrosstab: main_label (Excel) vs class mayoritaria (mask mapeada)")
display(ct)
agree = (ms["excel_idx"] == ms["majority_class"]).mean()
print(f"Coincidencia exacta Excel vs mayoria mask: {100*agree:.2f}% (interpretar with cuidado).")

# --- 4) Resumen numérico global (debe coincidir with analyze_masks.py in modo particion)
tot = ms["n_pixels"].sum()
for c in range(4):
    cn = CLASS_NAMES[c]
    s = ms[f"n_{cn}"].sum()
  print(f" Total pixels {cn}: {s:,} ({100*s/tot:.4f}% of the total)")


### Interpretation (noise vs. mapeo) and solutions

**What your numbers indicate**

- `min_pct_among_present` ≈ **0,000381%** in 512×512 (262 144 px) es **~1 pixel**. Las rows "minoritaria muy débil" are usually **a single pixel** distinto (borde, JPEG), no a global LUT failure.
- **Miles de masks with GG3/GG4 in (0, 0,1%)**: are usually **boundaries** o cuantización; el histograma by Excel muestra **solapamiento** — the label is **patch-level**, no pureza pixel a pixel.
- **Residual GG5** cerca de 0%: tras `clean_gg5_speckles` can leave a small tail; try increasing `gg5_gray_min` o `mask_clean_min_gg5_area`.

**Recommended actions**

1. **Criterio doble** for "noise": bajo **%** **y** low **pixel count** (e.g. `n < 16`, como la limpieza GG5).
2. **Celda siguiente**: barrido de umbrales (% and dual condition).
3. **Entrenamiento**: `gg5_gray_min` 175–180 o área mínima GG5 mayor (32, 64).
4. **Optional**: limpieza by componentes for **GG3/GG4** (analogous a GG5) o morfología by class.

**No** esperes alta coincidencia Excel vs mayoría in mask (tumor focal vs estroma).


In [ ]:
# Requires the previous cell (`ms` definido)

ex = int(ms["n_pixels"].iloc[0]) if len(ms) else 262144
print("Pixeles by parche (referencia):", ex)
print(" -> 1 pixel =", f"{100.0/ex:.6f}", "% of the parche")
print(" -> 0.1% of the parche ~", max(1, int(round(0.001 * ex))), "pixels\n")

rows_pct = []
for thr in [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0]:
    row = {"pct_thr": thr}
    for c in range(4):
        cn = CLASS_NAMES[c]
        n = int(((ms[f"n_{cn}"] > 0) & (ms[f"pct_{cn}"] < thr)).sum())
        row[cn] = n
    rows_pct.append(row)
tab_pct = pd.DataFrame(rows_pct)
print("Mascaras with class presente and pct < umbral (only relativo):")
display(tab_pct)

rows_dual = []
for thr in [0.1, 0.5, 1.0]:
    for nmax in [4, 8, 16, 32, 64]:
        row = {"pct_lt": thr, "n_lt": nmax}
        for c in range(4):
            cn = CLASS_NAMES[c]
            n = int(
                (
                    (ms[f"n_{cn}"] > 0)
                    & (ms[f"pct_{cn}"] < thr)
                    & (ms[f"n_{cn}"] < nmax)
                ).sum()
            )
            row[cn] = n
        rows_dual.append(row)
tab_dual = pd.DataFrame(rows_dual)
print("\nMascaras with 0 < pct < umbral% Y n_class < n_max pixels:")
display(tab_dual.sort_values(["pct_lt", "n_lt"]))

print("\nMascaras with 0 < n_class < 16 (sin condicion de %):")
for c in range(4):
    cn = CLASS_NAMES[c]
    k = int(((ms[f"n_{cn}"] > 0) & (ms[f"n_{cn}"] < 16)).sum())
  print(f" {cn}: {k:,}")


## Histograms of the **full** mask (grayscale raw)

Por grupo según `main_label` in Excel. Si hay muchos patches, cans bajar `MAX_PATCHES_PER_GROUP`.

In [ ]:
MAX_PATCHES_PER_GROUP: int | None = None  # ej. 800 para quick test


def hist_by_names(names: list[str]) -> np.ndarray:
    return accumulate_hist_full_image(names, max_patches=MAX_PATCHES_PER_GROUP)


groups = {
    "NC": meta.loc[meta["main_label"] == "NC", "image_name"].tolist(),
    "G3": meta.loc[meta["main_label"] == "G3", "image_name"].tolist(),
    "G4": meta.loc[meta["main_label"] == "G4", "image_name"].tolist(),
    "G5": meta.loc[meta["main_label"] == "G5", "image_name"].tolist(),
}
hists_raw: dict[str, np.ndarray] = {}
for lab, names in groups.items():
    if not names:
        continue
    hists_raw[lab] = hist_by_names(names)

x = np.arange(256)
fig, ax = plt.subplots(figsize=(11, 4))
for lab, h in hists_raw.items():
    ax.plot(x, np.log1p(h), label=lab, alpha=0.85)
for b in LUT_BOUNDARIES:
    ax.axvline(b, color="k", linestyle="--", alpha=0.35, linewidth=1)
ax.set_xlabel("Grayscale value (decoded mask)")
ax.set_ylabel("log(1 + recuento de pixels)")
ax.set_title("Global histogram por etiqueta Excel (todos los pixels del parche)")
ax.legend()
ax.set_xlim(0, 255)
plt.tight_layout()
plt.show()

## GG4 in Excel: **G4C=1** vs **G4C=0** (histograma of the full patch)

Misma métrica que arriba: todos los pixels de cada mask, agrupados by subtype.

In [ ]:
g4 = meta["main_label"] == "G4"
names_g4_crib = meta.loc[g4 & (meta["g4c"] == 1), "image_name"].tolist()
names_g4_plain = meta.loc[g4 & (meta["g4c"] == 0), "image_name"].tolist()
print(f"Parches G4 with G4C=1: {len(names_g4_crib)} | G4 with G4C=0: {len(names_g4_plain)}")

h_crib = hist_by_names(names_g4_crib) if names_g4_crib else np.zeros(256)
h_plain = hist_by_names(names_g4_plain) if names_g4_plain else np.zeros(256)

x = np.arange(256)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(x, np.log1p(h_crib), label="G4 + G4C=1 (cribriform)", color="darkgreen")
ax.plot(x, np.log1p(h_plain), label="G4 + G4C=0", color="steelblue")
for b in LUT_BOUNDARIES:
    ax.axvline(b, color="k", linestyle="--", alpha=0.35)
ax.set_xlabel("Grayscale value")
ax.set_ylabel("log(1 + count)")
ax.set_title("Parches etiquetados G4 en Excel: cribriform vs no cribriform")
ax.legend()
ax.set_xlim(0, 255)
plt.tight_layout()
plt.show()

## Only pixels that the current LUT classifies como **GG4** (class 2)

Here NC/G3/G5 pixels are ignored NC/G3/G5 within the patch: only raw intensities de regiones que **already** fall in `[85, DEFAULT_GG5_GRAY_MIN)` (e.g. `[85,170)` with default mapping) with la LUT vigente. This helps check whether G4C+ and G4C− separate in de grayscale **within **GG4**.

In [ ]:
pix_g4_crib = accumulate_gg4_pixels_raw(names_g4_crib)
pix_g4_plain = accumulate_gg4_pixels_raw(names_g4_plain)


def hist_from_pixels(pixels: np.ndarray) -> np.ndarray:
    if pixels.size == 0:
        return np.zeros(256)
    return np.bincount(pixels.astype(np.int64), minlength=256).astype(np.float64)


hc = hist_from_pixels(pix_g4_crib)
hp = hist_from_pixels(pix_g4_plain)

x = np.arange(256)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(x, np.log1p(hc), label="Píxeles LUT=GG4 | parche G4C=1", color="darkgreen")
ax.plot(x, np.log1p(hp), label="Píxeles LUT=GG4 | parche G4C=0", color="steelblue")
for b in LUT_BOUNDARIES:
    ax.axvline(b, color="k", linestyle="--", alpha=0.35)
ax.axvspan(85, DEFAULT_GG5_GRAY_MIN, alpha=0.08, color="orange", label=f"Rango LUT GG4 [85,{DEFAULT_GG5_GRAY_MIN})")
ax.set_xlim(0, 255)
ax.set_xlabel("Grayscale value raw")
ax.set_ylabel("log(1 + count)")
ax.set_title("Only pixels con MASK_LUT → GG4 (2)")
ax.legend()
plt.tight_layout()
plt.show()

if pix_g4_crib.size and pix_g4_plain.size:
    print(
        f"Media grayscale (only pixels LUT=GG4) | G4C=1: {pix_g4_crib.mean():.2f} | G4C=0: {pix_g4_plain.mean():.2f}"
    )

## Visualization: image + mask (grayscale, LUT, overlay)

Colormap LUT: negro=NC, colores for GG3/GG4/GG5.

In [ ]:
CMAP_LUT = ListedColormap(["#1a1a1a", "#4daf4a", "#377eb8", "#e41a1c"])  # NC, G3, G4, G5


def show_patch_row(name: str, title: str = ""):
    img = read_image_rgb(name)
    m = read_mask_gray(name)
    if img is None or m is None:
    print("Falta imagen o mask:", name)
        return
    mapped = MASK_LUT[m]
    fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
    axes[0].imshow(img)
    axes[0].set_title("RGB")
    axes[1].imshow(m, cmap="gray", vmin=0, vmax=255)
    axes[1].set_title("Máscara grayscale (bruto)")
    axes[2].imshow(mapped, cmap=CMAP_LUT, vmin=0, vmax=3)
    axes[2].set_title("Después de MASK_LUT")
    ov = img.copy().astype(np.float32) / 255.0
    color = np.zeros((*mapped.shape, 3), dtype=np.float32)
    color[mapped == 1] = [0.2, 0.8, 0.2]
    color[mapped == 2] = [0.2, 0.4, 0.9]
    color[mapped == 3] = [0.95, 0.2, 0.2]
    ov = 0.55 * ov + 0.45 * color
    axes[3].imshow(np.clip(ov, 0, 1))
    axes[3].set_title("Overlay")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(title or name[:60] + "…")
    plt.tight_layout()
    plt.show()


def sample_names(names: list[str], k: int = 3) -> list[str]:
    if not names:
        return []
    if len(names) <= k:
        return names
    return list(RNG.choice(names, size=k, replace=False))


print("--- Ejemplos G4 + G4C=1 ---")
for n in sample_names(names_g4_crib, 3):
    show_patch_row(n, title=f"G4 cribriform | {n[:50]}…")

print("--- Ejemplos G4 + G4C=0 ---")
for n in sample_names(names_g4_plain, 3):
    show_patch_row(n, title=f"G4 no cribriform | {n[:50]}…")

## Summary to tune the LUT

- If grayscale peaks **per class** overlap strongly between `[43,85)`, `[85,GG5_min)`, and `[GG5_min,255]` (con `GG5_min=DEFAULT_GG5_GRAY_MIN`, e.g. 170), there will be inherent confusion; la LUT only can **split** el eje de grayscale in intervalos.
- **G4C** is a clinical/annotation subtype; in 4 classes both map to **GG4**. Si cribriform and non-cribriform **no** se separan in intensidad within de `[85,GG5_min)`, do not expect una LUT de 4 classes que distinga G4C sin otra fuente de información.
- Next step: probar umbrales alternativos only tras inspeccionar estos histogramas and overlap with **GG3** and **GG5**.